## Configuración para poder importar desde el src/*

In [9]:
import os
import sys
from pathlib import Path
ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [10]:
import json

from common.common_types import LayoutElement
from common.data_storage import DataStorage


paths = DataStorage.find_json_paths()
dataset:list[list[LayoutElement]] = []
for path in paths:
    with open(path) as f:
        dataset.append(json.load(f))

print(dataset[0])


[{'label': 'FIELD_KEY_ID', 'text': 'R.U.C.:', 'source': 'digital', 'confidence': 1.0, 'page': 1, 'bbox': [291.0, 26.250022888183594, 330.33599853515625, 42.7380256652832], 'normalized_bbox': [489, 31, 555, 50]}, {'label': 'FIELD_VALUE_ID', 'text': '0990017514001', 'source': 'digital', 'confidence': 1.0, 'page': 1, 'bbox': [356.0, 24.79997444152832, 457.19195556640625, 44.03597640991211], 'normalized_bbox': [598, 29, 768, 52]}, {'label': 'O', 'text': 'NO', 'source': 'image_ocr', 'confidence': 0.97025927901268, 'page': 1, 'bbox': [27.5748502994012, 34.4616442499934, 70.95808383233532, 59.25299515595307], 'normalized_bbox': [46, 40, 119, 70]}, {'label': 'O', 'text': 'TIENE', 'source': 'image_ocr', 'confidence': 0.9901392936706543, 'page': 1, 'bbox': [80.59880239520959, 33.084346977440084, 168.7425149700599, 61.31894106478305], 'normalized_bbox': [135, 39, 283, 72]}, {'label': 'O', 'text': 'LOGO', 'source': 'image_ocr', 'confidence': 0.995796725153923, 'page': 1, 'bbox': [160.4790419161676

In [11]:
all_labels = set()
for doc in dataset:
    for e in doc:
        all_labels.add(e["label"])

label_list = sorted(list(all_labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Total facturas: {len(dataset)}")
print(f"Etiquetas: {label_list}")

Total facturas: 10
Etiquetas: ['FIELD_KEY_ADDRESS', 'FIELD_KEY_AMOUNT', 'FIELD_KEY_DATE', 'FIELD_KEY_EMAIL', 'FIELD_KEY_ID', 'FIELD_KEY_NAME', 'FIELD_KEY_TEXT', 'FIELD_VALUE_ADDRESS', 'FIELD_VALUE_AMOUNT', 'FIELD_VALUE_DATE', 'FIELD_VALUE_EMAIL', 'FIELD_VALUE_ID', 'FIELD_VALUE_NAME', 'FIELD_VALUE_TEXT', 'HEADER_PRODUCT_CODE', 'HEADER_PRODUCT_CODE_AUX', 'HEADER_PRODUCT_DETAIL', 'HEADER_PRODUCT_DISCOUNT', 'HEADER_PRODUCT_NAME', 'HEADER_PRODUCT_PRICE', 'HEADER_PRODUCT_QUANTITY', 'HEADER_PRODUCT_SUBSIDY', 'HEADER_PRODUCT_TOTAL', 'HEADER_PRODUCT_WITHOUT_SUBSIDY', 'ITEM_PRODUCT_CODE', 'ITEM_PRODUCT_CODE_AUX', 'ITEM_PRODUCT_DETAIL', 'ITEM_PRODUCT_DISCOUNT', 'ITEM_PRODUCT_NAME', 'ITEM_PRODUCT_PRICE', 'ITEM_PRODUCT_QUANTITY', 'ITEM_PRODUCT_SUBSIDY', 'ITEM_PRODUCT_TOTAL', 'ITEM_PRODUCT_WITHOUT_SUBSIDY', 'O']


## Preparación

In [12]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
from PIL import Image

def prepare_document(elements):
    words = [e["text"] for e in elements]
    boxes = [e["normalized_bbox"] for e in elements]
    labels = [label2id[e["label"]] for e in elements]
    return words, boxes, labels



def encode_document(elements):
    words, boxes, labels = prepare_document(elements)
    image = Image.new("RGB", (1000, 1000), color=255)
    encoding = processor(
        images=image,
        text=words,
        boxes=boxes,
        word_labels=labels,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    return encoding




## Entrenamiento

In [14]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
import torch

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="./model-output",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    save_steps=50,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [16]:
from torch.utils.data import Dataset as TorchDataset

class InvoiceDataset(TorchDataset):
    def __init__(self, documents):
        self.documents = documents

    def __getitem__(self, idx):
        encoding = encode_document(self.documents[idx])
        return {k: v.squeeze(0) for k, v in encoding.items()}

    def __len__(self):
        return len(self.documents)
    
split = int(len(dataset) * 0.8)

train_data = dataset[:split]
val_data = dataset[split:]
# train_data = dataset
# val_data = dataset
# print(len(train_data))

train_dataset = InvoiceDataset(train_data)
val_dataset = InvoiceDataset(val_data)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")



train_dataset = InvoiceDataset(train_data)
val_dataset = InvoiceDataset(val_data)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Train: 8 | Val: 2


  0%|          | 0/40 [00:00<?, ?it/s]c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 10%|█         | 4/40 [00:02<00:20,  1.76it/s]

{'eval_loss': 3.0451934337615967, 'eval_runtime': 0.1413, 'eval_samples_per_second': 14.158, 'eval_steps_per_second': 7.079, 'epoch': 1.0}


c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 20%|██        | 8/40 [00:06<00:24,  1.31it/s]

{'eval_loss': 2.678915500640869, 'eval_runtime': 0.1437, 'eval_samples_per_second': 13.918, 'eval_steps_per_second': 6.959, 'epoch': 2.0}


...


100%|██████████| 40/40 [00:38<00:00,  1.26it/s]

{'eval_loss': 1.3114497661590576, 'eval_runtime': 0.1249, 'eval_samples_per_second': 16.008, 'eval_steps_per_second': 8.004, 'epoch': 10.0}


100%|██████████| 40/40 [00:40<00:00,  1.01s/it]

{'train_runtime': 40.2049, 'train_samples_per_second': 1.99, 'train_steps_per_second': 0.995, 'train_loss': 2.0408390283584597, 'epoch': 10.0}


TrainOutput(global_step=40, training_loss=2.0408390283584597, metrics={'train_runtime': 40.2049, 'train_samples_per_second': 1.99, 'train_steps_per_second': 0.995, 'total_flos': 21238863052800.0, 'train_loss': 2.0408390283584597, 'epoch': 10.0})